# 09. E3: реальные данные, `extra` и `emo`

Те же 34 признака без пола, те же три модели, пять методов и настройки, что в E1 и E2.

- **Нулевой эталон:** перестановки цели на фиксированном разбиении 80/20 (seed 0), протокол
  тот же, что у наблюдаемой модели (`null_importances`: цель перемешивается отдельно
  в обучении и в тесте, seed модели свой у каждой перестановки). Каждый признак сравнивается
  со своим нулевым распределением.
- **Устойчивость:** повторная 5-кратная CV (5×5 = 25 фолдов), у каждого фолда свой seed модели.

**Персональные данные:** только агрегаты по признакам и повторам. Локальные объяснения
не сохраняются и не выводятся. Цель стандартизуется по всем строкам (среднее 0, SD 1),
важности поэтому в единицах SD черты.

Кеш: `results/e3_*.csv`, дописывается по блокам (черта, модель). Пересчёт: `E3_RECOMPUTE=1`.
Для проверки с сокращённым числом прогонов задайте `E3_SMOKE=1` перед запуском Jupyter.

In [ ]:
import os
import sys
import time
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.model_selection import RepeatedKFold, train_test_split

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from xai_consistency import explain as E  # noqa: E402
from xai_consistency import metrics as M  # noqa: E402
from xai_consistency.data import CONTINUOUS, FEATURES_LOG, FEATURES_OTHER, load_real  # noqa: E402
from xai_consistency.models import MODELS  # noqa: E402

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 400)

TRAITS = ("extra", "emo")
SMOKE = os.environ.get("E3_SMOKE") == "1"
N_PERM = 6 if SMOKE else int(os.environ.get("E3_PERM", 100))
CV_SPLITS, CV_REPEATS = 5, (1 if SMOKE else 5)
TEST_SIZE, SPLIT_SEED = 0.2, 0
K = 6  # top-k, как в E2
ALPHA = 0.05
N_JOBS = min(6, os.cpu_count() or 1)
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
if SMOKE:
    FIGURES = FIGURES / "e3_smoke"
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = {k: RESULTS / f"e3{'_smoke' if SMOKE else ''}_{k}.csv" for k in ("importance", "runs")}
METHODS = ["pi_test", "shap_test", "lime_test", "pdp_test", "ale_test", "shap_train", "lime_train"]
TEST_METHODS = METHODS[:5]
GROUP = ["shap_test", "lime_test", "pdp_test", "ale_test"]
METHOD_COLORS = {"pi": "#2a78d6", "shap": "#eb6834", "lime": "#1baf7a", "pdp": "#eda100", "ale": "#e87ba4"}

DATA = {}
for t in TRAITS:
    X, y = load_real(t)
    DATA[t] = (X, (y - y.mean()) / y.std())
FEATURES = list(DATA[TRAITS[0]][0].columns)
print("Строк:", len(DATA["extra"][0]), "признаков:", len(FEATURES), "из них непрерывных:", len(CONTINUOUS))

## 1. Один прогон: наблюдаемый, перестановка или фолд CV

In [ ]:
def split_for(trait, kind, idx):
    """Разбиение и seed модели для прогона. kind: obs, null, cv."""
    X, y = DATA[trait]
    if kind in ("obs", "null"):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=SPLIT_SEED)
        if kind == "null":  # как в explain.null_importances
            r = np.random.default_rng(SPLIT_SEED + 10_000 + idx)
            y_tr, y_te = r.permutation(y_tr), r.permutation(y_te)
            return X_tr, X_te, y_tr, y_te, SPLIT_SEED + idx
        return X_tr, X_te, y_tr, y_te, SPLIT_SEED
    folds = list(RepeatedKFold(n_splits=CV_SPLITS, n_repeats=CV_REPEATS, random_state=0).split(X))
    tr, te = folds[idx]
    return X.iloc[tr], X.iloc[te], y[tr], y[te], 100 + idx


def run_job(trait, model_name, kind, idx):
    t0 = time.perf_counter()
    X_tr, X_te, y_tr, y_te, seed = split_for(trait, kind, idx)
    res = E.fit_and_explain(model_name, X_tr, y_tr, X_te, y_te, log_cols=FEATURES_LOG,
                            other_cols=FEATURES_OTHER, curve_features=CONTINUOUS, seed=seed)
    key = {"trait": trait, "model": model_name, "kind": kind, "idx": idx}
    rows = []
    for (m, part), ex in res.explanations.items():
        for f in res.features:
            rows.append({**key, "method": f"{m}_{part}", "feature": f,
                         "importance": ex.importance.get(f, np.nan), "sign": ex.sign.get(f, np.nan)})
    run = {**key, "seed": seed, "r2_train": res.r2_train, "r2_test": res.r2_test,
           "seconds": time.perf_counter() - t0}
    return rows, run


def jobs_for(trait, model_name):
    return ([(trait, model_name, "obs", 0)] + [(trait, model_name, "null", i) for i in range(N_PERM)]
            + [(trait, model_name, "cv", i) for i in range(CV_SPLITS * CV_REPEATS)])

## 2. Настройки запуска

Для принудительного пересчёта задайте `E3_RECOMPUTE=1` перед запуском Jupyter.

## 3. Все прогоны (или кеш)

In [ ]:
def read_cache(path):
    """Строка "null" (kind перестановки) не должна читаться как NaN."""
    return pd.read_csv(path, keep_default_na=False, na_values=[""], low_memory=False)


recompute = os.environ.get("E3_RECOMPUTE") == "1"
tables = {k: read_cache(p) for k, p in CACHE.items()} if not recompute and all(p.exists() for p in CACHE.values()) else {}
done = set(tables["runs"][["trait", "model", "kind", "idx"]].itertuples(index=False, name=None)) if tables else set()
t_all = time.perf_counter()
for model_name in ("rf", "hgb", "ridge"):
    for trait in TRAITS:
        jobs = [j for j in jobs_for(trait, model_name) if j not in done]
        if not jobs:
            continue
        t0 = time.perf_counter()
        out = Parallel(n_jobs=N_JOBS)(delayed(run_job)(*j) for j in jobs)
        print(f"{trait} × {model_name}: {len(jobs)} прогонов за {(time.perf_counter() - t0) / 60:.1f} мин")
        new = {"importance": pd.DataFrame([r for o in out for r in o[0]]), "runs": pd.DataFrame([o[1] for o in out])}
        for k, df in new.items():
            tables[k] = pd.concat([tables[k], df], ignore_index=True) if k in tables else df
            tables[k].to_csv(CACHE[k], index=False)
print(f"Расчёт: {(time.perf_counter() - t_all) / 60:.1f} мин")
imp, runs = tables["importance"], tables["runs"]
print("Прогонов:", runs.groupby(["trait", "model", "kind"]).size().unstack().to_string())
print("Среднее время прогона в процессе, с:", runs.groupby("model")["seconds"].mean().round(1).to_dict())

# Широкие таблицы: (черта, модель, kind, idx, метод) × признаки
# pivot, а не pivot_table(dropna=False): тот строит полное произведение уровней индекса,
# и у cv появились бы пустые строки с idx перестановок
IDX = ["trait", "model", "kind", "idx", "method"]
wide = imp.pivot(index=IDX, columns="feature", values="importance").reindex(columns=FEATURES).sort_index()
signs = imp.pivot(index=IDX, columns="feature", values="sign").reindex(columns=FEATURES).sort_index()
n_rows = wide.groupby(level=["trait", "model", "kind", "method"]).size()
assert n_rows.xs("cv", level="kind").eq(CV_SPLITS * CV_REPEATS).all() and n_rows.xs("null", level="kind").eq(N_PERM).all()


def get(trait, model_name, kind, method):
    """Таблица повторы × признаки (только признаки, у которых метод определён)."""
    t = wide.loc[(trait, model_name, kind)].xs(method, level="method")
    return t.dropna(axis=1, how="all")

## 4. Качество моделей: наблюдаемый R² против перестановок и на CV

In [ ]:
rows = []
for (trait, model_name), g in runs.groupby(["trait", "model"]):
    obs = g.loc[g["kind"] == "obs", "r2_test"].iloc[0]
    null = g.loc[g["kind"] == "null", "r2_test"].to_numpy()
    cv = g.loc[g["kind"] == "cv", "r2_test"]
    rows.append({"trait": trait, "model": model_name, "R² тест (разбиение 0)": obs,
                 "нуль: среднее": null.mean(), "нуль: 95%": np.quantile(null, 0.95),
                 "p": (1 + (null >= obs).sum()) / (1 + len(null)),
                 "CV: среднее": cv.mean(), "CV: SD по фолдам": cv.std(),
                 "CV: доля фолдов > 0": (cv > 0).mean()})
quality = pd.DataFrame(rows).set_index(["trait", "model"])
print(quality.round(3).to_string())

## 5. Важность каждого признака против его собственного нуля

Для каждого (черта, модель, метод, признак): процентиль наблюдаемой важности среди
перестановок, эмпирическое p = (1 + #{нуль ≥ набл.})/(1 + n), BH по признакам метода
(34 у PI, SHAP, LIME; 20 непрерывных у PDP и ALE). Основной критерий «выделился» —
q < 0.05 по хвостовому p (нормальная аппроксимация нуля, у PI на исходной шкале,
у остальных на log): у эмпирического p при 100 перестановках не хватает разрешения
(минимум 1/101, а BH для одного открытия из 34 требует p ≤ 0.0015).

In [ ]:
def null_test(trait, model_name, method):
    obs = get(trait, model_name, "obs", method).iloc[0]
    null = get(trait, model_name, "null", method)[obs.index]
    p_emp = M.null_pvalues(obs, null)
    p_tail = M.null_tail_pvalues(obs, null, log=not method.startswith("pi"))
    return pd.DataFrame({"obs": obs, "null_mean": null.mean(), "null_q95": null.quantile(0.95),
                         "percentile": (null.to_numpy() < obs.to_numpy()[None, :]).mean(axis=0) * 100,
                         "p_emp": p_emp, "q_emp": M.bh_fdr(p_emp), "p_tail": p_tail, "q_tail": M.bh_fdr(p_tail),
                         "z": M.null_zscore(obs, null)})


tests = pd.concat({(t, m, meth): null_test(t, m, meth) for t in TRAITS for m in MODELS for meth in METHODS},
                  names=["trait", "model", "method", "feature"]).reset_index()

### 5.1 Калибровка хвостового p на самих перестановках

Каждая перестановка по очереди считается «наблюдаемой», нуль — остальные. При верной
аппроксимации доли p < 0.05 и p < 0.01 близки к 0.05 и 0.01.

In [ ]:
def calibration(trait, model_name, method):
    null = get(trait, model_name, "null", method)
    ps = []
    for i in range(len(null)):
        rest = null.drop(index=null.index[i])
        ps.append(M.null_tail_pvalues(null.iloc[i], rest, log=not method.startswith("pi")).to_numpy())
    ps = np.concatenate(ps)
    return {"p<0.05": np.mean(ps < 0.05), "p<0.01": np.mean(ps < 0.01), "p<0.001": np.mean(ps < 0.001)}


cal = pd.DataFrame({(t, m, meth): calibration(t, m, meth) for t in TRAITS for m in MODELS for meth in TEST_METHODS}).T
cal.index.names = ["trait", "model", "method"]
print("Доля перестановок с хвостовым p ниже порога (ожидается 0.05, 0.01, 0.001):")
print(cal.groupby(level="method").agg(["mean", "max"]).round(3).to_string())

### 5.1б Калибровка отдельно для флагов и непрерывных признаков

Добавлено после первого разбора. Без сигнала PI флага у деревьев часто ровно 0 (модель флаг
не использует) с крошечным разбросом, и нормальная аппроксимация на таком нуле может давать
ложно малые p. Порог 0.002 близок к тому, что нужно BH для первых открытий из 34 (0.0015–0.003).

In [ ]:
FLAGS = [f for f in FEATURES if f not in CONTINUOUS]


def loo_tail_p(null, method):
    """p каждой перестановки против остальных: перестановки × признаки."""
    return pd.DataFrame([M.null_tail_pvalues(null.iloc[i], null.drop(index=null.index[i]),
                                             log=not method.startswith("pi")).to_numpy()
                         for i in range(len(null))], columns=null.columns)


rows = []
for t in TRAITS:
    for m in MODELS:
        for meth in ["pi_test", "shap_test", "lime_test"]:
            P = loo_tail_p(get(t, m, "null", meth), meth)
            for typ, cols in [("флаги", FLAGS), ("непрерывные", CONTINUOUS)]:
                v = P[cols].to_numpy().ravel()
                rows.append({"trait": t, "model": m, "method": meth, "type": typ,
                             **{f"p<{a}": np.mean(v < a) for a in (0.05, 0.01, 0.002)}})
cal_type = pd.DataFrame(rows)
print("Доля перестановок с хвостовым p ниже порога, по типу признака (среднее по двум чертам):")
print(cal_type.groupby(["model", "method", "type"])[["p<0.05", "p<0.01", "p<0.002"]].mean().round(4).to_string())

In [ ]:
print("extra × HGB, три флага: наблюдаемая важность против 100 перестановок")
rows = []
for meth in ["pi_test", "shap_test", "lime_test"]:
    obs, null = get("extra", "hgb", "obs", meth).iloc[0], get("extra", "hgb", "null", meth)
    tst = tests.set_index(["trait", "model", "method", "feature"]).loc[("extra", "hgb", meth)]
    for f in ["has_university", "profile_block", "personal_block"]:
        rows.append({"method": meth, "feature": f, "перестановок ≥ набл.": int((null[f] >= obs[f]).sum()),
                     "p_emp": tst.loc[f, "p_emp"], "p_tail": tst.loc[f, "p_tail"], "q_tail": tst.loc[f, "q_tail"],
                     "доля нулевых ровно 0": float((null[f].abs() < 1e-12).mean())})
print(pd.DataFrame(rows).round(4).to_string(index=False))

### 5.1в extra × HGB: 1000 перестановок только для PI

PI без SHAP и LIME дешёвая, поэтому для HGB у `extra` берём 1000 перестановок (тот же
протокол и seed: первые 100 совпадают с основными). Эмпирическое p с разрешением 1/1001
и BH по 34 признакам работают без хвостовой аппроксимации.

In [ ]:
N_PI = 12 if SMOKE else 1000
CACHE_PI = RESULTS / f"e3{'_smoke' if SMOKE else ''}_hgb_pi_{N_PI}.csv"


def run_pi_only(trait, model_name, idx):
    X_tr, X_te, y_tr, y_te, seed = split_for(trait, "null", idx)
    res = E.fit_and_explain(model_name, X_tr, y_tr, X_te, y_te, log_cols=FEATURES_LOG, other_cols=FEATURES_OTHER,
                            curve_features=CONTINUOUS, seed=seed, methods=("pi",), splits=())
    return {"idx": idx, **res[("pi", "test")].importance.to_dict()}


if CACHE_PI.exists() and not recompute:
    pi_big = read_cache(CACHE_PI)
else:
    t0 = time.perf_counter()
    pi_big = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(run_pi_only)("extra", "hgb", i) for i in range(N_PI)))
    print(f"HGB, только PI: {N_PI} перестановок за {(time.perf_counter() - t0) / 60:.1f} мин")
    pi_big.to_csv(CACHE_PI, index=False)
pi_big = pi_big.set_index("idx")[FEATURES]
same = get("extra", "hgb", "null", "pi_test").sort_index()
print("Совпадение первых перестановок с основным расчётом, макс. |разница|:",
      float((pi_big.loc[same.index[:N_PERM]] - same.to_numpy()).abs().max().max()))
obs = get("extra", "hgb", "obs", "pi_test").iloc[0]
p1000 = M.null_pvalues(obs, pi_big)
pi_tab = pd.DataFrame({"перестановок ≥ набл.": (pi_big >= obs).sum(), "p_emp_1000": p1000, "q_emp_1000": M.bh_fdr(p1000),
                       "q_tail_100": tests.set_index(["trait", "model", "method", "feature"]).loc[("extra", "hgb", "pi_test"), "q_tail"]})
print(f"extra × HGB × PI, {N_PI} перестановок: признаки с q_emp < 0.05 или q_tail(100) < 0.05")
print(pi_tab[(pi_tab["q_emp_1000"] < ALPHA) | (pi_tab["q_tail_100"] < ALPHA)].sort_values("p_emp_1000").round(4).to_string())
print("Калибровка хвостового p на 1000 перестановках (флаги / непрерывные):")
P = loo_tail_p(pi_big.iloc[:300], "pi_test")  # 300 перестановок: оценка доли до ~0.001 по 300·14 значениям
for typ, cols in [("флаги", FLAGS), ("непрерывные", CONTINUOUS)]:
    v = P[cols].to_numpy().ravel()
    print(f"  {typ}:", {a: round(float(np.mean(v < a)), 4) for a in (0.05, 0.01, 0.002)})

In [ ]:
print("Число признаков, которые выделились (q_tail < 0.05), и при эмпирическом p:")
cnt = tests.assign(tail=tests["q_tail"] < ALPHA, emp=tests["q_emp"] < ALPHA, p05=tests["p_emp"] < ALPHA,
                   above_max=tests["percentile"] == 100)
print(cnt.pivot_table(index=["trait", "model"], columns="method", values="tail", aggfunc="sum")[METHODS].to_string())
print("\nq_emp < 0.05:")
print(cnt.pivot_table(index=["trait", "model"], columns="method", values="emp", aggfunc="sum")[METHODS].to_string())
print("\nНаблюдаемая важность выше всех перестановок (число признаков):")
print(cnt.pivot_table(index=["trait", "model"], columns="method", values="above_max", aggfunc="sum")[METHODS].to_string())
print("\nНескорректированное p_emp < 0.05 (число признаков; ожидание при нуле 34·0.05 = 1.7, у PDP и ALE 1.0):")
print(cnt.pivot_table(index=["trait", "model"], columns="method", values="p05", aggfunc="sum")[METHODS].to_string())

In [ ]:
print("Какие признаки выделились (q_tail < 0.05): число комбинаций модель × метод (тест, 5 методов × 3 модели = 15):")
hits = tests[(tests["q_tail"] < ALPHA) & tests["method"].isin(TEST_METHODS)]
print(hits.groupby(["trait", "feature"]).size().sort_values(ascending=False).to_string())

In [ ]:
print("Признаки, выделившиеся хотя бы раз: z относительно нуля и процентиль (тест):")
top_feats = {t: hits.loc[hits["trait"] == t, "feature"].value_counts().index.tolist() for t in TRAITS}
for t in TRAITS:
    if not top_feats[t]:
        print(f"{t}: ни одного")
        continue
    sub = tests[(tests["trait"] == t) & tests["feature"].isin(top_feats[t]) & tests["method"].isin(TEST_METHODS)]
    print(f"\n{t}: z")
    print(sub.pivot_table(index=["feature", "model"], columns="method", values="z")[TEST_METHODS].round(1).to_string())
    print(f"{t}: q_tail")
    print(sub.pivot_table(index=["feature", "model"], columns="method", values="q_tail")[TEST_METHODS].round(3).to_string())

In [ ]:
print("Минимальное q_tail по признакам у каждой комбинации (насколько близко к порогу):")
print(tests.pivot_table(index=["trait", "model"], columns="method", values="q_tail", aggfunc="min")[METHODS].round(3).to_string())

### 5.2 Масштаб модели (Ridge)

RidgeCV на перемешанной цели выбирает большой α и сжимает все коэффициенты, на настоящей
цели α меньше. Тогда абсолютная важность шумовых признаков у наблюдаемой модели выше,
чем у нулевых, и признак «отличается от нуля» только из-за масштаба. Проверка: тот же
тест на доле важности (важность / сумма по признакам метода).

In [ ]:
def null_test_share(trait, model_name, method):
    obs = get(trait, model_name, "obs", method).iloc[0]
    null = get(trait, model_name, "null", method)[obs.index]
    if method.startswith("pi"):
        return None
    obs, null = obs / obs.sum(), null.div(null.sum(axis=1), axis=0)
    p = M.null_tail_pvalues(obs, null)
    return pd.DataFrame({"q_tail_share": M.bh_fdr(p)})


share = pd.concat({(t, m, meth): r for t in TRAITS for m in MODELS for meth in METHODS
                   if (r := null_test_share(t, m, meth)) is not None},
                  names=["trait", "model", "method", "feature"]).reset_index()
print("Число выделившихся признаков по доле важности (q < 0.05):")
print(share.assign(hit=share["q_tail_share"] < ALPHA).pivot_table(
    index=["trait", "model"], columns="method", values="hit", aggfunc="sum").to_string())
ridge_alpha = {}
for t in TRAITS:
    from xai_consistency.models import make_pipeline
    X_tr, X_te, y_tr, y_te, _ = split_for(t, "obs", 0)
    pipe = make_pipeline("ridge", FEATURES_LOG, FEATURES_OTHER).fit(X_tr, y_tr)
    alphas_null = []
    for i in range(20):
        _, _, y_tr_n, _, _ = split_for(t, "null", i)
        alphas_null.append(make_pipeline("ridge", FEATURES_LOG, FEATURES_OTHER).fit(X_tr, y_tr_n).named_steps["model"].alpha_)
    ridge_alpha[t] = {"alpha наблюдаемой": pipe.named_steps["model"].alpha_, "alpha нуля, медиана (20 перест.)": np.median(alphas_null)}
print(pd.DataFrame(ridge_alpha).T.round(1))

### 5.3 Ridge: нуль с α, зафиксированным на наблюдаемом

Добавлено после прогонов. Нулевые RidgeCV выбирают α = 10⁴ (верхняя граница сетки),
наблюдаемая у `extra` — около 264. Тест по доле важности для Ridge оказался слабым
(не находит даже друзей). Честнее сравнивать с нулём той же модели: перестановки
с α, равным наблюдаемому. У `emo` наблюдаемый α тоже 10⁴, там пересчёт не нужен.

In [ ]:
CACHE_FIXED = RESULTS / f"e3{'_smoke' if SMOKE else ''}_ridge_fixed_alpha.csv"


def run_fixed_alpha(trait, idx, alpha):
    """Нулевой прогон Ridge с фиксированным α (подмена make_model внутри процесса)."""
    from sklearn.linear_model import RidgeCV

    from xai_consistency import models as Mo
    original = Mo.make_model
    Mo.make_model = lambda name, random_state=42: RidgeCV(alphas=[alpha])
    try:
        rows, _ = run_job(trait, "ridge", "null", idx)
    finally:
        Mo.make_model = original  # процессы joblib переиспользуются
    return rows


N_FIXED = 12 if SMOKE else 1000  # сначала было 100, затем добрано до 1000 (раздел 5.4)
fixed = read_cache(CACHE_FIXED) if CACHE_FIXED.exists() and not recompute else pd.DataFrame(columns=["idx"])
todo = [i for i in range(N_FIXED) if i not in set(fixed["idx"].astype(int))]
if todo:
    a_obs = ridge_alpha["extra"]["alpha наблюдаемой"]
    t0 = time.perf_counter()
    out = Parallel(n_jobs=N_JOBS)(delayed(run_fixed_alpha)("extra", i, a_obs) for i in todo)
    print(f"Ridge с α = {a_obs:.1f}: {len(todo)} перестановок за {(time.perf_counter() - t0) / 60:.1f} мин")
    fixed = pd.concat([fixed, pd.DataFrame([r for o in out for r in o])], ignore_index=True)
    fixed.to_csv(CACHE_FIXED, index=False)
fixed["idx"] = fixed["idx"].astype(int)
fw_all = fixed.pivot(index=["idx", "method"], columns="feature", values="importance").reindex(columns=FEATURES)
fw = fw_all[fw_all.index.get_level_values("idx") < N_PERM]  # первые 100, как в первом разборе
rows = {}
for meth in METHODS:
    obs = get("extra", "ridge", "obs", meth).iloc[0]
    null = fw.xs(meth, level="method")[obs.index]
    p = M.null_tail_pvalues(obs, null, log=not meth.startswith("pi"))
    rows[meth] = M.bh_fdr(p)
q_fixed = pd.DataFrame(rows)
print("extra × Ridge, нуль с фиксированным α: признаки с q_tail < 0.05")
print({m: q_fixed.index[q_fixed[m] < ALPHA].tolist() for m in METHODS})
print("q_tail для признаков, выделившихся хотя бы у одного метода:")
print(q_fixed[(q_fixed < ALPHA).any(axis=1)].round(3).to_string())

У Ridge mean |SHAP| ∝ |β|, а нуль для |β| близок к полунормальному. У его логарифма длинный
левый хвост, SD на log-шкале большая, и хвостовой тест теряет мощность. Нормальная
аппроксимация на исходной шкале, наоборот, слишком либеральна (калибровка ниже). Поэтому
для Ridge с фиксированным α смотрим и на признаки выше всех перестановок.

In [ ]:
for meth in ["shap_test", "pi_test"]:
    obs = get("extra", "ridge", "obs", meth).iloc[0]
    null = fw.xs(meth, level="method")[obs.index]
    p_raw = M.null_tail_pvalues(obs, null, log=False)
    t = pd.DataFrame({"набл./медиана нуля": obs / null.median(), "выше всех": obs > null.max(),
                      "p_emp": M.null_pvalues(obs, null), "p_tail_log": M.null_tail_pvalues(obs, null, log=not meth.startswith("pi")),
                      "p_tail_raw": p_raw, "q_tail_raw": M.bh_fdr(p_raw)}).sort_values("p_emp").head(6)
    print(meth)
    print(t.round(4).to_string())
null = fw.xs("shap_test", level="method")
for log in (True, False):
    ps = np.concatenate([M.null_tail_pvalues(null.iloc[i], null.drop(index=null.index[i]), log=log).to_numpy()
                         for i in range(len(null))])
    print(f"Калибровка, SHAP Ridge с фикс. α, {'log' if log else 'исходная шкала'}:",
          {t: round(float(np.mean(ps < t)), 4) for t in (0.05, 0.01, 0.001)})

### 5.4 Ridge с фиксированным α: 1000 перестановок, эмпирическое p и BH

При 1000 перестановках минимальное p = 1/1001 ≈ 0.001, и BH по 34 признакам может
отвергать гипотезы без хвостовой аппроксимации. Сравниваем с выводом по первым 100.

In [ ]:
rows = {}
for meth in METHODS:
    obs = get("extra", "ridge", "obs", meth).iloc[0]
    null_all = fw_all.xs(meth, level="method")[obs.index]
    p_all = M.null_pvalues(obs, null_all)
    null_100 = fw.xs(meth, level="method")[obs.index]
    rows[meth] = pd.DataFrame({"n_ge": (null_all >= obs).sum(), "p_emp": p_all, "q_emp": M.bh_fdr(p_all),
                               "above_max_100": obs > null_100.max()})
r1000 = pd.concat(rows, names=["method", "feature"])
n_all = fw_all.index.get_level_values("idx").nunique()
print(f"extra × Ridge, α фиксирован, {n_all} перестановок: признаки с q_emp < 0.05")
print({m: r1000.loc[m].index[r1000.loc[m, "q_emp"] < ALPHA].tolist() for m in METHODS})
print("\nПо 100 перестановкам «выше всех» было:", {m: r1000.loc[m].index[r1000.loc[m, "above_max_100"]].tolist() for m in METHODS})
hit = r1000[(r1000["q_emp"] < ALPHA) | r1000["above_max_100"]].reset_index()
print("\nПодробно (тест):")
print(hit[hit["method"].isin(TEST_METHODS)].pivot_table(index="feature", columns="method", values=["n_ge", "q_emp"])
      .round(4).to_string())

### 5.5 extra × RF и HGB: 1000 перестановок для PI, SHAP, PDP, ALE

Добавлено при закрытии E3. У RF друзья и лайки выделялись только как «выше всех 100
перестановок», а BH по 34 признакам на таком разрешении не проходит. Тот же протокол и seed,
что в основном расчёте (первые 100 перестановок совпадают), без LIME (около часа на модель)
и без SHAP на обучении. Оценка времени: 0.55 с на перестановку RF и 0.2 с у HGB на 6 процессах.

In [ ]:
N_TREE = 12 if SMOKE else 1000
CACHE_TREE = RESULTS / f"e3{'_smoke' if SMOKE else ''}_trees_1000.csv"
TREE_METHODS = ["pi_test", "shap_test", "pdp_test", "ale_test"]


def run_tree_null(model_name, idx):
    X_tr, X_te, y_tr, y_te, seed = split_for("extra", "null", idx)
    res = E.fit_and_explain(model_name, X_tr, y_tr, X_te, y_te, log_cols=FEATURES_LOG, other_cols=FEATURES_OTHER,
                            curve_features=CONTINUOUS, seed=seed, methods=("pi", "pdp", "ale", "shap"), splits=("test",))
    return [{"model": model_name, "idx": idx, "method": f"{m}_{s}", "feature": f, "importance": ex.importance.get(f, np.nan)}
            for (m, s), ex in res.explanations.items() for f in res.features]


tree = read_cache(CACHE_TREE) if CACHE_TREE.exists() and not recompute else pd.DataFrame(columns=["model", "idx"])
for m in ("rf", "hgb"):
    have = set(tree.loc[tree["model"] == m, "idx"].astype(int))
    todo = [i for i in range(N_TREE) if i not in have]
    if todo:
        t0 = time.perf_counter()
        out = Parallel(n_jobs=N_JOBS)(delayed(run_tree_null)(m, i) for i in todo)
        print(f"extra × {m}, без LIME: {len(todo)} перестановок за {(time.perf_counter() - t0) / 60:.1f} мин")
        tree = pd.concat([tree, pd.DataFrame([r for o in out for r in o])], ignore_index=True)
        tree.to_csv(CACHE_TREE, index=False)
tree["idx"] = tree["idx"].astype(int)
tw = tree.pivot(index=["model", "method", "idx"], columns="feature", values="importance").reindex(columns=FEATURES)
for m in ("rf", "hgb"):
    for meth in TREE_METHODS:
        last_shared_idx = min(N_PERM, N_TREE) - 1
        a = tw.loc[(m, meth)].loc[:last_shared_idx].dropna(axis=1, how="all")
        b = get("extra", m, "null", meth).sort_index().loc[a.index, a.columns]
        assert np.allclose(a.to_numpy(), b.to_numpy(), atol=1e-9), (m, meth)
print(f"Перестановок, совпавших с основным расчётом: {min(N_PERM, N_TREE)} (RF и HGB, 4 метода)")

rows = {}
for m in ("rf", "hgb"):
    for meth in TREE_METHODS:
        obs = get("extra", m, "obs", meth).iloc[0]
        null = tw.loc[(m, meth)][obs.index]
        p = M.null_pvalues(obs, null)
        rows[(m, meth)] = pd.DataFrame({"n_ge": (null >= obs).sum(), "p_emp": p, "q_emp": M.bh_fdr(p)})
t1000 = pd.concat(rows, names=["model", "method", "feature"])
print(f"extra × RF, HGB, {N_TREE} перестановок: признаки с q_emp < 0.05")
print({k: t1000.loc[k].index[t1000.loc[k, "q_emp"] < ALPHA].tolist() for k in rows})
print("Признаки с q_emp < 0.1 хотя бы у одной комбинации:")
near = t1000[t1000["q_emp"] < 0.1].index.get_level_values("feature").unique()
print(t1000[t1000.index.get_level_values("feature").isin(near)].reset_index()
      .pivot_table(index="feature", columns=["model", "method"], values=["n_ge", "q_emp"]).round(3).to_string())

## 6. Устойчивость: 5×5 CV и seed моделей

In [ ]:
def pairwise_rho(t):
    """Средний ρ Спирмена важностей между всеми парами повторов."""
    vals = [M.rank_agreement(t.iloc[i], t.iloc[j])["spearman"] for i, j in combinations(range(len(t)), 2)]
    return np.nanmean(vals)


rows = []
for t in TRAITS:
    for m in MODELS:
        for meth in METHODS:
            cv, null = get(t, m, "cv", meth), get(t, m, "null", meth)
            rows.append({"trait": t, "model": m, "method": meth, "CV": pairwise_rho(cv),
                         "нуль": pairwise_rho(null.iloc[:CV_SPLITS * CV_REPEATS])})
stab = pd.DataFrame(rows)
print("Средний ρ важностей между повторами (CV — 25 фолдов; нуль — 25 перестановок на одном разбиении):")
print(stab.pivot_table(index=["trait", "model"], columns="method", values=["CV", "нуль"])
      .reindex(columns=METHODS, level=1).round(2).to_string())

In [ ]:
def topk_share(t):
    ranks = t.rank(axis=1, ascending=False)
    return (ranks <= K).mean()


rows = []
for t in TRAITS:
    for m in MODELS:
        for meth in TEST_METHODS:
            cv, null = get(t, m, "cv", meth), get(t, m, "null", meth)
            r_cv = cv.rank(axis=1, ascending=False)
            for f in cv.columns:
                rows.append({"trait": t, "model": m, "method": meth, "feature": f,
                             "top6_cv": (r_cv[f] <= K).mean(), "top6_null": topk_share(null)[f],
                             "rank_median": r_cv[f].median(), "rank_iqr": r_cv[f].quantile(0.75) - r_cv[f].quantile(0.25),
                             "imp_cv_mean": cv[f].mean(), "imp_cv_sd": cv[f].std()})
topk = pd.DataFrame(rows)
for t in TRAITS:
    sub = topk[topk["trait"] == t]
    order = sub.groupby("feature")["top6_cv"].mean().sort_values(ascending=False).index[:8]
    print(f"\n{t}: доля фолдов CV, где признак в top-6 (в скобках — доля перестановок), 8 самых частых:")
    s = sub[sub["feature"].isin(order)]
    tab = (s["top6_cv"].round(2).astype(str) + " (" + s["top6_null"].round(2).astype(str) + ")")
    print(s.assign(v=tab).pivot_table(index=["feature"], columns=["model", "method"], values="v", aggfunc="first")
          .reindex(order).to_string())

In [ ]:
for t in TRAITS:
    sub = topk[(topk["trait"] == t) & topk["method"].isin(["pi_test", "shap_test"])]
    order = sub.groupby("feature")["top6_cv"].mean().sort_values(ascending=False).index[:5]
    print(f"\n{t}: медианный ранг и IQR ранга по 25 фолдам, CV важности (SD/среднее):")
    s = sub[sub["feature"].isin(order)].assign(cv_coef=lambda d: d["imp_cv_sd"] / d["imp_cv_mean"].abs())
    print(s.pivot_table(index=["feature"], columns=["model", "method"], values=["rank_median", "rank_iqr", "cv_coef"])
          .reindex(order).round(1).to_string())

## 7. Знаки выделившихся признаков

In [ ]:
for t in TRAITS:
    if not top_feats[t]:
        continue
    print(f"\n{t}: доля фолдов CV, где знак эффекта +1 (SHAP, LIME, PDP, ALE):")
    rows = []
    for m in MODELS:
        for meth in GROUP:
            s = signs.loc[(t, m, "cv")].xs(meth, level="method")
            for f in top_feats[t]:
                if f in s and s[f].notna().any():
                    rows.append({"feature": f, "model": m, "method": meth, "plus": (s[f] > 0).mean()})
    print(pd.DataFrame(rows).pivot_table(index="feature", columns=["model", "method"], values="plus").round(2).to_string())

## 8. Согласие методов между собой

Как в E2: пары методов на 20 непрерывных признаках. Группа SHAP, LIME, PDP, ALE и PI против
остальных. Отдельно на фолдах CV (настоящая цель) и на перестановках (нуль).

In [ ]:
cont = [c for c in CONTINUOUS if c in FEATURES]
rows = []
for (t, m, kind, idx), g in wide.groupby(level=["trait", "model", "kind", "idx"]):
    if kind == "obs":
        continue
    g = g.droplevel(["trait", "model", "kind", "idx"])
    for a, b in combinations(TEST_METHODS, 2):
        r = M.rank_agreement(g.loc[a], g.loc[b], cont)
        rows.append({"trait": t, "model": m, "kind": kind, "idx": idx, "a": a, "b": b, "spearman": r["spearman"]})
agree = pd.DataFrame(rows)
agree["group"] = np.where(agree["a"].eq("pi_test") | agree["b"].eq("pi_test"), "PI — остальные", "внутри SHAP, LIME, PDP, ALE")
grp = agree.groupby(["trait", "model", "kind", "group"])["spearman"].mean().unstack(["kind", "group"])
print(grp.round(2).to_string())
print("\nПо парам (CV):")
print(agree[agree["kind"] == "cv"].assign(pair=lambda d: d["a"] + "–" + d["b"])
      .pivot_table(index=["trait", "model"], columns="pair", values="spearman").round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, len(TRAITS), figsize=(10, 3.6), sharey=True)
for ax, t in zip(axes, TRAITS):
    xs = np.arange(len(MODELS))
    for k, (g, color) in enumerate([("внутри SHAP, LIME, PDP, ALE", "#eb6834"), ("PI — остальные", "#2a78d6")]):
        for j, (kind, alpha) in enumerate([("cv", 1.0), ("null", 0.45)]):
            sub = agree[(agree["trait"] == t) & (agree["group"] == g) & (agree["kind"] == kind)]
            per = sub.groupby(["model", "idx"])["spearman"].mean()
            mean = per.groupby("model").mean().reindex(MODELS)
            q = per.groupby("model").quantile([0.1, 0.9]).unstack().reindex(MODELS)
            pos = xs + (k * 2 + j - 1.5) * 0.17
            ax.errorbar(pos, mean, yerr=[mean - q[0.1], q[0.9] - mean], fmt="o", color=color, alpha=alpha,
                        ms=5, capsize=2, label=f"{g}, {'CV' if kind == 'cv' else 'нуль'}")
    ax.axhline(0, color="#888888", lw=1, ls=":")
    ax.set_xticks(xs)
    ax.set_xticklabels(MODELS)
    ax.set_title(t)
    ax.grid(alpha=0.2, axis="y")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("ρ Спирмена важностей (20 непрерывных)")
axes[0].legend(frameon=False, fontsize=7, loc="lower left")
fig.suptitle("E3: согласие методов на настоящей цели (CV) и на перестановках (нуль); 10–90% повторов")
fig.tight_layout()
fig.savefig(FIGURES / "e3_agreement_groups.png", dpi=110)
plt.show()

## 9. Обучение против теста

In [ ]:
rows = []
for t in TRAITS:
    for m in MODELS:
        for kind in ("cv", "null"):
            for meth in ("shap", "lime"):
                a, b = get(t, m, kind, f"{meth}_train"), get(t, m, kind, f"{meth}_test")
                rows.append({"trait": t, "model": m, "kind": kind, "method": meth,
                             "rho": np.nanmean([M.rank_agreement(a.iloc[i], b.iloc[i])["spearman"] for i in range(len(a))])})
print("ρ рангов обучение–тест (среднее по повторам):")
print(pd.DataFrame(rows).pivot_table(index=["trait", "model"], columns=["kind", "method"], values="rho").round(2).to_string())

## 10. Локальное против глобального (вопрос исследования 2)

Добавлено после первого разбора. SHAP и LIME на тесте (200 объектов) у обеих черт, на тех же
прогонах, что выше: наблюдаемый, 100 перестановок, 25 фолдов CV. Пересчитываются только SHAP
и LIME на тесте, с теми же seed (проверка: mean |SHAP| совпадает с сохранённой важностью).
**Сохраняются только агрегаты по объектам** одного прогона, локальные значения не пишутся.

Метрики прогона:
- **знак вклада** друзей и лайков: доля объектов, у которых знак вклада (φ у SHAP, коэффициент ×
  отклонение у LIME) совпадает с глобальным знаком признака. Для монотонного эффекта вклад
  отрицателен у объектов ниже фона, поэтому эта доля по построению близка к доле объектов
  по одну сторону от фона и сама по себе о согласии мало говорит;
- **направление эффекта**: SHAP — sign(φ)·sign(z − среднее фона), LIME — знак локального
  наклона. Доля объектов, где направление совпадает с глобальным знаком;
- **ρ по объектам** между |локальными вкладами| и глобальным порядком: собственным (среднее |вклада|
  того же метода) и PI на тесте. Медиана и квартили по объектам;
- **друзья в локальном топ-3**: на экстраверсии для всех трёх моделей считаем долю тестовых профилей
  каждого фолда, где число друзей входит в тройку из 34 признаков по абсолютному вкладу.
  При равенстве вкладов сохраняем исходный порядок столбцов, как в `metrics.top_k`;
- **разброс**: SD локальных вкладов признака / среднее |вклада| (у LIME ещё SD наклона /
  |средний наклон|) у друзей и лайков и медиана по всем признакам.

In [ ]:
CACHE_LOCAL = RESULTS / f"e3{'_smoke' if SMOKE else ''}_local.csv"
LOCAL_FEATS = ["vk_counter_friends", "vk_post_avg_likes"]
SHORT = {"vk_counter_friends": "friends", "vk_post_avg_likes": "likes"}
TOP3_RUNS = {("extra", model, "cv") for model in MODELS}


def iqr_stats(v, name):
    v = np.asarray(v, float)
    return {f"{name}_median": np.nanmedian(v), f"{name}_q25": np.nanpercentile(v, 25), f"{name}_q75": np.nanpercentile(v, 75)}


def run_local(trait, model_name, kind, idx):
    X_tr, X_te, y_tr, y_te, seed = split_for(trait, kind, idx)
    res = E.fit_and_explain(model_name, X_tr, y_tr, X_te, y_te, log_cols=FEATURES_LOG, other_cols=FEATURES_OTHER,
                            curve_features=CONTINUOUS, seed=seed, methods=("shap", "lime"), splits=("test",))
    Z = res.samples["test"]
    pi = wide.loc[(trait, model_name, kind, idx, "pi_test")]
    out = []
    for meth in ("shap", "lime"):
        ex = res[(meth, "test")]
        contrib = ex.local if meth == "shap" else ex.info["contrib"]
        if meth == "shap":
            direction = np.sign(contrib) * np.sign(Z - res.background.mean())
        else:
            direction = np.sign(ex.local)
        row = {"trait": trait, "model": model_name, "kind": kind, "idx": idx, "method": meth,
               "mean_abs_check": float(contrib.abs().mean()["vk_counter_friends"]) if meth == "shap" else np.nan}
        for f in LOCAL_FEATS:
            g = ex.sign[f]
            c, d = np.sign(contrib[f]), direction[f]
            row[f"{SHORT[f]}_global_sign"] = g
            row[f"{SHORT[f]}_contrib_sign"] = float((c[c != 0] == g).mean()) if g != 0 else np.nan
            row[f"{SHORT[f]}_direction"] = float((d[d != 0] == g).mean()) if g != 0 else np.nan
            row[f"{SHORT[f]}_spread"] = float(contrib[f].std() / contrib[f].abs().mean())
            if meth == "lime":
                row[f"{SHORT[f]}_slope_spread"] = float(ex.local[f].std() / abs(ex.local[f].mean()))
        spread_all = contrib.std() / contrib.abs().mean().replace(0, np.nan)
        row["spread_all_median"] = float(spread_all.median())
        absl = contrib.abs()
        if (trait, model_name, kind) in TOP3_RUNS:
            ranks = absl.rank(axis=1, method="first", ascending=False)
            row["friends_top3_share"] = float(ranks["vk_counter_friends"].le(3).mean())
        own = absl.mean()
        row.update(iqr_stats([M.rank_agreement(absl.iloc[i], own)["spearman"] for i in range(len(absl))], "rho_own"))
        row.update(iqr_stats([M.rank_agreement(absl.iloc[i], pi)["spearman"] for i in range(len(absl))], "rho_pi"))
        out.append(row)
    return out


local_jobs = [j for t in TRAITS for m in MODELS for j in jobs_for(t, m)]
run_columns = ["trait", "model", "kind", "idx"]
local_cache_keys = run_columns + ["method"]
local_tab = read_cache(CACHE_LOCAL) if CACHE_LOCAL.exists() and not recompute else pd.DataFrame(columns=local_cache_keys)
done_l = set(local_tab[run_columns].astype({"idx": int}).itertuples(index=False, name=None))
# Старый кеш не содержит топ-3. Пересчитываем только фолды, нужные для этой метрики.
top3_rows = local_tab.dropna(subset=["friends_top3_share"]) if "friends_top3_share" in local_tab else local_tab.iloc[:0]
top3_methods = top3_rows.groupby(run_columns)["method"].nunique()
done_top3 = set(top3_methods[top3_methods == 2].index)
todo = [j for j in local_jobs if j not in done_l or (j[:3] in TOP3_RUNS and j not in done_top3)]
if todo:
    todo.sort(key=lambda j: {"rf": 0, "hgb": 1, "ridge": 2}[j[1]])
    t0 = time.perf_counter()
    out = Parallel(n_jobs=N_JOBS)(delayed(run_local)(*j) for j in todo)
    print(f"Локальные метрики: {len(todo)} прогонов за {(time.perf_counter() - t0) / 60:.1f} мин")
    new_rows = pd.DataFrame([r for o in out for r in o])
    local_tab = pd.concat([local_tab, new_rows], ignore_index=True).drop_duplicates(local_cache_keys, keep="last")
    local_tab.to_csv(CACHE_LOCAL, index=False)
local_tab["idx"] = local_tab["idx"].astype(int)
chk = local_tab[local_tab["method"] == "shap"].merge(
    imp[(imp["method"] == "shap_test") & (imp["feature"] == "vk_counter_friends")][["trait", "model", "kind", "idx", "importance"]],
    on=["trait", "model", "kind", "idx"])
print("Проверка: макс. |mean |SHAP| друзей − сохранённая важность| =", float((chk["mean_abs_check"] - chk["importance"]).abs().max()))

In [ ]:
LOCAL_METRICS = ["friends_contrib_sign", "likes_contrib_sign", "friends_direction", "likes_direction",
                 "rho_own_median", "rho_own_q25", "rho_own_q75", "rho_pi_median",
                 "friends_spread", "likes_spread", "spread_all_median", "friends_slope_spread", "likes_slope_spread"]


def local_summary(metric):
    """Наблюдаемый прогон, среднее по CV и нуль (среднее и 5–95%), процентиль наблюдаемого в нуле."""
    rows = []
    for (t, m, meth), g in local_tab.groupby(["trait", "model", "method"]):
        if metric not in g or g[metric].isna().all():
            continue
        obs = g.loc[g["kind"] == "obs", metric].iloc[0]
        null = g.loc[g["kind"] == "null", metric].dropna()
        cv = g.loc[g["kind"] == "cv", metric]
        rows.append({"trait": t, "model": m, "method": meth, "набл.": obs, "CV": cv.mean(),
                     "нуль": null.mean(), "нуль 5%": null.quantile(0.05), "нуль 95%": null.quantile(0.95),
                     "процентиль набл.": (null < obs).mean() * 100, "процентиль CV": (null < cv.mean()).mean() * 100})
    return pd.DataFrame(rows).set_index(["method", "trait", "model"]).sort_index()


for metric in LOCAL_METRICS:
    print(f"\n{metric}:")
    print(local_summary(metric).round(2).to_string())

print("\nДрузья в локальном топ-3, экстраверсия: доли по тестовым фолдам, %")
top3 = local_tab.groupby(["model", "method"])["friends_top3_share"].agg(["count", "mean", "min", "max"])
top3 = top3[top3["count"] > 0]
assert len(top3) == 2 * len(MODELS) and top3["count"].eq(CV_SPLITS * CV_REPEATS).all()
print(top3[["mean", "min", "max"]].mul(100).round(1).to_string())

In [ ]:
print("Глобальный знак друзей и лайков: доля прогонов со знаком +1")
print(local_tab.groupby(["trait", "model", "method", "kind"])[["friends_global_sign", "likes_global_sign"]]
      .agg(lambda v: (v > 0).mean()).unstack("kind").round(2).to_string())

## Выводы

1. **`extra`** (1000 перестановок, эмпирическое p и BH; хвостовой тест отброшен как либеральный):
   у RF друзья и лайки у PI, SHAP, PDP и ALE (q 0.010–0.017), у HGB друзья у всех четырёх методов, лайки только
   у PDP. Флаги у деревьев не значимы. Ridge с RidgeCV нельзя сравнивать с нулём (нулевые α = 10⁴);
   с фиксированным α — друзья, лайки, gifts, у SHAP и LIME has_university. LIME деревьев только
   на 100 перестановках.
2. **`emo`:** средний R² на CV отрицательный. Данных 100 перестановок недостаточно для вывода об отдельных признаках.
3. **Раскол методов** тот же, что в E1 и E2: SHAP, LIME, PDP и ALE согласны (0.81–0.97)
   у обеих черт и в нуле, PI с ними согласен только у `extra`.
4. **Устойчивость рангов SHAP и LIME** между фолдами высокая и у `emo` (0.74–0.85 у деревьев),
   поэтому она не доказывает сигнал.
5. **Локально** `extra` почти не отличается от `emo` и перестановок: порядок вкладов у объекта,
   их разброс и знак вклада одинаковы. Отличаются согласие с PI, направление SHAP (на границе
   нуля) и однородность наклонов друзей у LIME. На экстраверсии число друзей входит
   в локальный топ-3 у большинства тестовых профилей в каждом фолде RF и HGB по SHAP и LIME.
   У Ridge доля по обоим методам в среднем составляет около половины.

## Графики E3

In [ ]:
from scipy import stats

C = {"pi": "#2a78d6", "shap": "#eb6834", "lime": "#1baf7a", "pdp": "#eda100", "ale": "#e87ba4",
     "lime_cat": "#6250d6", "rf": "#2a78d6", "hgb": "#eb6834", "ridge": "#1baf7a",
     "extra": "#2a78d6", "emo": "#eb6834", "null": "#9a9893", "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df"}
MARK = {"pi": "o", "shap": "s", "lime": "^", "pdp": "D", "ale": "v", "rf": "o", "hgb": "s", "ridge": "^"}
NAME = {"pi": "PI", "shap": "SHAP", "lime": "LIME", "pdp": "PDP", "ale": "ALE",
        "rf": "RF", "hgb": "HGB", "ridge": "Ridge"}
SHORT = {"vk_counter_friends": "друзья", "vk_post_avg_likes": "лайки", "has_university": "has_university",
         "vk_counter_audios": "аудио", "vk_can_see_audio": "can_see_audio"}

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.edgecolor": C["ink2"], "axes.labelcolor": C["ink"], "xtick.color": C["ink2"], "ytick.color": C["ink2"],
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": C["grid"],
    "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False, "lines.linewidth": 2,
    "lines.markersize": 6, "figure.dpi": 100, "savefig.bbox": "tight", "pdf.fonttype": 42,
})
def save(fig, name, title):
    fig.savefig(FIGURES / f"{name}.png", dpi=200)
    plt.show()
    plt.close(fig)
    print(f"{name}: {title}")


def z_scores(trait, model, method):
    """z наблюдаемой важности относительно нуля того же признака (extra деревья: 1000 перестановок)."""
    o = imp[(imp["trait"] == trait) & (imp["model"] == model) & (imp["kind"] == "obs") & (imp["method"] == method)]
    o = o.set_index("feature")["importance"]
    if trait == "extra" and model in ("rf", "hgb"):
        n = tree[(tree["model"] == model) & (tree["method"] == method)]
    else:
        n = imp[(imp["trait"] == trait) & (imp["model"] == model) & (imp["kind"] == "null") & (imp["method"] == method)]
    n = n.pivot(index="idx", columns="feature", values="importance")[o.index]
    return (o - n.mean()) / n.std()


fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.35]})
ax = axes[0]
rng = np.random.default_rng(0)
for i, mod in enumerate(["ridge", "rf", "hgb"]):
    for j, (lab, sel, col) in enumerate((("нуль", (runs["model"] == mod) & (runs["kind"] == "null") & (runs["trait"] == "extra"), C["null"]),
                                         ("emo", (runs["model"] == mod) & (runs["kind"] == "cv") & (runs["trait"] == "emo"), C["emo"]),
                                         ("extra", (runs["model"] == mod) & (runs["kind"] == "cv") & (runs["trait"] == "extra"), C["extra"]))):
        v = runs.loc[sel, "r2_test"].to_numpy()
        x = i + (j - 1) * 0.27 + rng.uniform(-0.07, 0.07, len(v))
        ax.scatter(x, v, s=9, color=col, alpha=0.55, lw=0, label=lab if i == 0 else None)
        ax.plot([i + (j - 1) * 0.27 - 0.1, i + (j - 1) * 0.27 + 0.1], [np.mean(v)] * 2, color=C["ink"], lw=1.6)
ax.axhline(0, color=C["ink2"], lw=0.8)
ax.set_xticks(range(3), ["Ridge", "RF", "HGB"])
ax.set_ylabel("R² на тесте")
ax.grid(axis="x", visible=False)
ax.legend(loc="lower left", ncol=3, markerscale=1.8)
ax.set_title(f"а) Качество: {CV_SPLITS * CV_REPEATS} фолдов CV и {N_PERM} перестановок", loc="left")

ax = axes[1]
pos = 0
labels = []
for mod in ("rf", "hgb"):
    for meth in ("pi_test", "shap_test"):
        for trait in ("extra", "emo"):
            z = z_scores(trait, mod, meth)
            y = pos + rng.uniform(-0.18, 0.18, len(z))
            ax.scatter(z.values, y, s=14, color=C[trait], alpha=0.75, lw=0)
            for f, dy in (("vk_counter_friends", 7), ("vk_post_avg_likes", -7)):
                if z[f] > 2.5:
                    ax.annotate(SHORT[f], (z[f], y[z.index.get_loc(f)]), xytext=(3, dy), textcoords="offset points",
                                fontsize=7, va="center", color=C["ink2"])
            labels.append(f"{trait}: {NAME[mod]}, {NAME[meth.split('_')[0]]}")
            pos += 1
ax.axvline(0, color=C["ink2"], lw=0.8)
ax.set_yticks(range(pos), labels)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("z важности относительно нуля этого признака")
ax.set_title("б) 34 признака: важность против перестановок", loc="left")
fig.suptitle("Друзья и лайки выделяются у extra", x=0.01, y=1.04, ha="left", fontsize=10.5)
fig.text(0.01, -0.06, "Нуль: перемешанная цель, то же разбиение 80/20. z — только для наглядности; "
         f"для важных признаков extra у деревьев — {N_TREE} перестановок, для emo — {N_PERM}.",
         fontsize=7.5, color=C["ink2"])
fig.tight_layout()
save(fig, "fig05_e3_null", "extra против emo против нуля: качество и важности (E3)")

### Устойчивость рангов между фолдами и перестановками

In [ ]:
W = imp[imp["method"].isin(TEST_METHODS)].pivot_table(index=["trait", "model", "kind", "idx", "method"],
                                                  columns="feature", values="importance")


def mean_pair_rho(block):
    block = block.dropna(axis=1)  # у PDP и ALE нет флагов
    r = stats.spearmanr(block.to_numpy().T)[0]
    return r[np.triu_indices(len(block), 1)].mean()


st = []
for (t, m, k, meth), g in W.groupby(level=["trait", "model", "kind", "method"]):
    if k == "obs" or (k == "null" and t != "extra"):
        continue
    g = g.iloc[:CV_SPLITS * CV_REPEATS]
    st.append({"trait": t, "model": m, "kind": k, "method": meth.split("_")[0], "rho": mean_pair_rho(g)})
st = pd.DataFrame(st)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, mod in zip(axes, ("rf", "hgb")):
    for j, (lab, sel) in enumerate((("нуль", (st["kind"] == "null")), ("emo, CV", (st["trait"] == "emo") & (st["kind"] == "cv")),
                                     ("extra, CV", (st["trait"] == "extra") & (st["kind"] == "cv")))):
        for i, meth in enumerate(["pi", "shap", "lime", "pdp", "ale"]):
            v = st[sel & (st["model"] == mod) & (st["method"] == meth)]["rho"].iloc[0]
            ax.bar(j + (i - 2) * 0.16, v, width=0.15, color=C[meth], label=NAME[meth] if j == 0 else None)
    ax.set_xticks(range(3), ["нуль", "emo, CV", "extra, CV"])
    ax.axhline(0, color=C["ink2"], lw=0.8)
    ax.grid(axis="x", visible=False)
    ax.set_title(NAME[mod], loc="left")
axes[0].set_ylabel("средний ρ важностей между прогонами")
axes[1].legend(loc="upper left", ncol=5, fontsize=7.5, bbox_to_anchor=(-1.1, -0.12))
fig.suptitle("Ранги SHAP и LIME устойчивы и без сигнала, устойчивость PI следует за сигналом", x=0.01, y=1.04, ha="left", fontsize=10.5)
fig.text(0.01, -0.2, f"E3. CV — {CV_SPLITS}×{CV_REPEATS}; нуль — столько же перестановок цели extra на одном разбиении. "
         "34 признака у PI, SHAP, LIME; 20 непрерывных у PDP и ALE.", fontsize=7.5, color=C["ink2"])
save(fig, "fig06_stability", "Устойчивость рангов: extra, emo и нуль (E3)")